In [ ]:
import jax
import jax.numpy as jnp
from jax import lax, jit, grad
from jax.experimental import mesh_utils
from jax.sharding import PositionalSharding
from functools import partial
import time

# -------------------------------------------------------------------------
# Configuration & Constants
# -------------------------------------------------------------------------
MAX_RECURSION_DEPTH    = 1_000_000
OPTIMAL_DEPTH_STEP     = 250_000
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE             = 50_000_000

# If you want to clamp the input to avoid NaNs, adjust here
VAL_CLAMP_LOW  = -100.0
VAL_CLAMP_HIGH =  100.0

# -------------------------------------------------------------------------
# 1) Dynamic pi & phi with scaling
#    -> "Ensure recursion grows proportionally to TPU-friendly execution."
# -------------------------------------------------------------------------
@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    """
    We add a dynamic scaling:
      - If depth is large, reduce the exponent faster,
        so we don't blow up at massive recursion counts.
    """
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)

    # Example: incorporate an adaptive scale so that for very large depth,
    # we get smaller 'phi_dyn' to keep values stable
    adapt = jnp.maximum(1.0, depth / 100_000.0)  # e.g. scale by depth above 100k
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor * adapt + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    """Normalizes depth scaling to prevent excessive iteration blow-ups."""
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return depth / (1 + jnp.log1p(depth + 1))

# -------------------------------------------------------------------------
# 2) Single 250k-step chunk with potential clamp to avoid NaNs.
# -------------------------------------------------------------------------
def single_chunk_fori_loop(x, scale_factor=1.0):
    """
    Runs 250k steps of sin/exp recursion in one fori_loop.
    Returns the final 'x' after 250k iterations.
    """
    def body_fn(i, val):
        pi_dyn  = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

        # Optionally clamp 'val' to reduce blow-ups
        safe_val = jnp.clip(val, VAL_CLAMP_LOW, VAL_CLAMP_HIGH)

        new_val  = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 1))
        return new_val

    steps = jnp.int32(OPTIMAL_DEPTH_STEP)
    return lax.fori_loop(0, steps, body_fn, x)

# -------------------------------------------------------------------------
# 3) Chunked recursion with debug prints & "dynamic phi" scaling
# -------------------------------------------------------------------------
@partial(jit, static_argnames=["total_depth", "scale_factor"])
def chunked_dppu_debug(x, total_depth, scale_factor=1.0):
    """
    Single JIT-compiled function that:
      - Splits total_depth into (total_depth // 250k) chunks
      - Each chunk calls single_chunk_fori_loop
      - Prints debug info about the partial results
    """
    iterations = total_depth // OPTIMAL_DEPTH_STEP

    def chunk_body(chunk_idx, val):
        new_val = single_chunk_fori_loop(val, scale_factor=scale_factor)
        mean_val = jnp.mean(new_val)

        # jax.debug.print prints host-side after each chunk
        jax.debug.print(
            "Chunk {chunk_idx} => mean(new_val)={mean_val:.6f}",
            chunk_idx=chunk_idx,
            mean_val=mean_val
        )
        return new_val

    final_x = lax.fori_loop(0, iterations, chunk_body, x)
    return final_x

# -------------------------------------------------------------------------
# 4) Utility functions
# -------------------------------------------------------------------------
def log_tpu_memory(msg=""):
    """
    Placeholder for logging TPU memory usage or device metrics.
    In actual practice, you might integrate with:
      - Cloud TPU profiling
      - Cloud Monitoring
      - TPU system logs
    or read from specialized APIs.
    """
    print(f"[MEM DEBUG] {msg} - (Memory usage not implemented in code)")

def precompile_recursion(depths=(250_000, 500_000, 1_000_000), scale_factor=0.5):
    """
    Forces XLA to compile the chunked recursion for specific depths before real runs.
    This avoids on-the-fly compilation overhead.
    """
    dummy_input = jnp.zeros((1_000,))  # small shape for fast compile
    for d in depths:
        print(f"Precompiling recursion for depth={d} with scale_factor={scale_factor}")
        _ = chunked_dppu_debug(dummy_input, total_depth=d, scale_factor=scale_factor).block_until_ready()

def process_with_larger_depths_debug(x, total_depth, scale_factor=0.5):
    """
    High-level function to call the single JIT-compiled chunked recursion.
    You can add logs, memory checks, etc. here as well.
    """
    return chunked_dppu_debug(x, total_depth=total_depth, scale_factor=scale_factor)

# -------------------------------------------------------------------------
# 5) Benchmark & Execution
# -------------------------------------------------------------------------
def run_tpu_benchmarks(batch_input, depths=(250_000, 500_000, 1_000_000), scale_factor=0.5, num_trials=2):
    """
    Runs benchmarks at the specified depths, records timing, etc.
    Demonstrates caching by repeating the same shapes multiple times.
    """
    results = []

    for depth in depths:
        times = []
        for trial_i in range(num_trials):
            log_tpu_memory(msg=f"Before depth={depth}, trial={trial_i}")
            start_time = time.time()

            output = process_with_larger_depths_debug(batch_input, depth, scale_factor)
            # Force device sync to measure full cost
            out_host = jax.device_get(output)

            elapsed = time.time() - start_time
            times.append(elapsed)

            log_tpu_memory(msg=f"After depth={depth}, trial={trial_i}")
            # A small statistic
            mean_val = float(jnp.mean(out_host))
            results.append({
                "depth": depth,
                "trial": trial_i,
                "time": elapsed,
                "mean_output": mean_val
            })

        # Print summary for this depth
        avg_time = sum(times) / len(times)
        print(f"\n🔥 TPU Benchmark (Depth={depth}, Scale={scale_factor}, Batch={BATCH_SIZE})")
        print(f"  Avg: {avg_time:.6f} sec | Min: {min(times):.6f} sec | Max: {max(times):.6f} sec")

    return results

# -------------------------------------------------------------------------
# Main script
# -------------------------------------------------------------------------
if __name__ == "__main__":
    # 1) TPU Sharding Setup
    devices = jax.devices()
    sharding = PositionalSharding(devices)

    # 2) Prepare the large batch input
    print("Allocating batch_input...")
    batch_input = jnp.linspace(0, 10, BATCH_SIZE)
    batch_input = jax.device_put(batch_input, sharding)

    # 3) Precompile for recursion depths to reduce compile overhead
    precompile_recursion(depths=[250_000, 500_000, 1_000_000], scale_factor=0.5)

    # 4) Run the benchmarks with caching.
    #    We do each depth 2 times to see if the second run is faster (cache effect).
    bench_results = run_tpu_benchmarks(
        batch_input,
        depths=[250_000, 500_000, 1_000_000],
        scale_factor=0.5,
        num_trials=2
    )

    # 5) Print final result details
    print("\nCollected Benchmark Results:")
    for r in bench_results:
        print(r)


Allocating batch_input...
Precompiling recursion for depth=250000 with scale_factor=0.5
Chunk 0 => mean(new_val)=0.000000
Precompiling recursion for depth=500000 with scale_factor=0.5
Chunk 0 => mean(new_val)=0.000000
Chunk 1 => mean(new_val)=0.000000
Precompiling recursion for depth=1000000 with scale_factor=0.5
Chunk 0 => mean(new_val)=0.000000
Chunk 1 => mean(new_val)=0.000000
Chunk 2 => mean(new_val)=0.000000
Chunk 3 => mean(new_val)=0.000000
[MEM DEBUG] Before depth=250000, trial=0 - (Memory usage not implemented in code)
Chunk 0 => mean(new_val)=0.000000
[MEM DEBUG] After depth=250000, trial=0 - (Memory usage not implemented in code)
[MEM DEBUG] Before depth=250000, trial=1 - (Memory usage not implemented in code)
Chunk 0 => mean(new_val)=0.000000
[MEM DEBUG] After depth=250000, trial=1 - (Memory usage not implemented in code)

🔥 TPU Benchmark (Depth=250000, Scale=0.5, Batch=50000000)
  Avg: 323.066965 sec | Min: 322.797108 sec | Max: 323.336822 sec
[MEM DEBUG] Before depth=50000